# Chapter 04 Companion Notebook: Optimization and Gradient Descent

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch04_Gradient_Descent.ipynb)

This notebook accompanies Chapter 04 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic marketing campaign data and transparent NumPy implementations. It requires no paid API, external dataset, downloaded model, or GPU.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Keep `FAST_MODE = True` during a class session. The notebook begins with loss functions, slopes, gradients, convexity, and the gradient descent update rule. It then trains a linear regression model from scratch, compares full-batch, stochastic, and mini-batch updates, demonstrates why feature scaling matters, and closes with practical optimizer extensions, early stopping, regularization, leakage controls, and troubleshooting.

## Why this matters (business framing)

Many analytics models are defined by an objective rather than by a direct formula for the final parameters. The model learns by repeatedly changing its parameters so that a loss becomes smaller. Gradient descent is the basic engine behind that process.

A business analyst does not need advanced calculus to use gradient descent responsibly, but the analyst must understand what the optimizer is trying to minimize, how the learning rate controls movement, why batches create noise, and why unscaled data can make a simple problem unnecessarily difficult. These choices affect training time, stability, reproducibility, and sometimes the credibility of the final result.

This notebook treats optimization as an observable workflow. Every experiment records the loss, gradient size, update count, validation behavior, and data boundary. The goal is not merely to make a loss decrease. The goal is to reach a useful solution efficiently without using future information or hiding unstable training behavior.

## Agenda

1. Setup and reproducibility
2. Business decision contract and synthetic campaign data
3. Regression and classification loss functions
4. Convexity, local minima, and saddle points
5. Derivatives, partial derivatives, and gradients
6. The gradient descent update rule and learning rate
7. Linear regression from scratch
8. Full-batch, stochastic, and mini-batch gradient descent
9. Feature scaling and leakage-safe preprocessing
10. Learning-rate schedules, momentum, and Adam
11. Regularization and early stopping
12. Troubleshooting and gradient checks
13. Governance and saved artifacts
14. Exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to compute and compare common regression and classification losses, explain why the negative gradient points downhill, approximate a gradient numerically, implement the gradient descent update rule, diagnose learning rates that are too small or too large, train linear regression with full-batch and mini-batch updates, quantify gradient noise across batch sizes, explain feature scaling through curvature and condition numbers, fit preprocessing on training data only, compare basic optimizers, use validation loss for early stopping, and document an optimization experiment so that another analyst can reproduce it.

## Connection map

Chapter 2 uses visualization to make optimization behavior visible. Chapter 5 develops the evaluation boundaries needed to decide whether a trained model generalizes. Chapter 6 applies gradient descent to linear regression, Chapter 7 adds regularization, Chapter 8 applies log loss to logistic regression, and Chapter 9 uses optimization ideas in support vector machines. Chapter 14 returns to the same update loop inside deep neural networks. Across all of these chapters, the recurring pattern is to define a loss, compute a gradient, choose an update rule, protect the validation and test boundaries, and monitor whether the procedure is converging.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
import json
import math
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("joblib", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 160)

SEED = 685
FAST_MODE = True


def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


reset_seeds()

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = BASE_DIR / "baai_ch04_gradient_descent_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_CAMPAIGNS = 600 if FAST_MODE else 1200
MAX_EPOCHS = 500 if FAST_MODE else 1000

print({
    "FAST_MODE": FAST_MODE,
    "N_CAMPAIGNS": N_CAMPAIGNS,
    "MAX_EPOCHS": MAX_EPOCHS,
    "OUT_DIR": str(OUT_DIR),
})

## Utility functions

The helpers below define losses, metrics, linear predictions, and numerical gradients. The regression objective is written as one-half of the mean squared error. The factor of one-half does not change the minimum, but it cancels the factor of two when the derivative is calculated.

In [ ]:
# ============================================================
# Utility functions
# ============================================================
def sigmoid(x):
    x = np.clip(np.asarray(x, dtype=float), -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


def squared_loss(residual):
    residual = np.asarray(residual, dtype=float)
    return 0.5 * residual ** 2


def absolute_loss(residual):
    return np.abs(np.asarray(residual, dtype=float))


def huber_loss(residual, delta=1.0):
    residual = np.asarray(residual, dtype=float)
    magnitude = np.abs(residual)
    return np.where(
        magnitude <= delta,
        0.5 * residual ** 2,
        delta * (magnitude - 0.5 * delta),
    )


def binary_log_loss(y_true, probability):
    y_true = np.asarray(y_true, dtype=float)
    probability = np.clip(np.asarray(probability, dtype=float), 1e-12, 1 - 1e-12)
    return -(y_true * np.log(probability) + (1 - y_true) * np.log(1 - probability))


def half_mse(y_true, prediction):
    error = np.asarray(prediction) - np.asarray(y_true)
    return float(0.5 * np.mean(error ** 2))


def add_intercept(X):
    X = np.asarray(X, dtype=float)
    return np.column_stack([np.ones(len(X)), X])


def linear_gradient(X, y, weights, l2=0.0):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    residual = X @ weights - y
    gradient = X.T @ residual / len(y)
    if l2 > 0:
        penalty = weights.copy()
        penalty[0] = 0.0
        gradient = gradient + l2 * penalty
    return gradient


def regression_metrics(y_true, prediction):
    y_true = np.asarray(y_true, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, prediction))),
        "mae": float(mean_absolute_error(y_true, prediction)),
        "r2": float(r2_score(y_true, prediction)),
    }


def finite_difference_gradient(objective, parameters, epsilon=1e-6):
    parameters = np.asarray(parameters, dtype=float)
    approximation = np.zeros_like(parameters)
    for j in range(len(parameters)):
        plus = parameters.copy()
        minus = parameters.copy()
        plus[j] += epsilon
        minus[j] -= epsilon
        approximation[j] = (objective(plus) - objective(minus)) / (2 * epsilon)
    return approximation


def standardized_to_raw_coefficients(weights, scaler, feature_names):
    weights = np.asarray(weights, dtype=float)
    raw_slopes = weights[1:] / scaler.scale_
    raw_intercept = weights[0] - np.sum(weights[1:] * scaler.mean_ / scaler.scale_)
    rows = [{"term": "intercept", "coefficient": raw_intercept}]
    rows.extend(
        {"term": name, "coefficient": value}
        for name, value in zip(feature_names, raw_slopes)
    )
    return pd.DataFrame(rows)

## 2. Business decision contract and synthetic campaign data

The unit of analysis is one marketing campaign observed at its launch date. The continuous target is revenue during the next seven days, measured in thousands of dollars. The intended use is campaign planning and budget allocation, not causal attribution.

The data contain advertising spend, audience size, discount rate, seasonality, and a competitor price index. Their scales differ sharply on purpose. Advertising spend is measured in dollars, audience size in people, and discount as a proportion. A gradual time trend creates a realistic future distribution shift, so preprocessing must be fitted on the training period only.

In [ ]:
# ============================================================
# 2.1 Decision contract
# ============================================================
decision_contract_df = pd.DataFrame([
    {"component": "unit of analysis", "definition": "one campaign at its launch date"},
    {"component": "target", "definition": "revenue during the next 7 days, in thousands of dollars"},
    {"component": "prediction moment", "definition": "immediately before campaign launch"},
    {"component": "intended use", "definition": "forecast campaign revenue for planning and budget review"},
    {"component": "not intended for", "definition": "causal claims about the effect of advertising or discounts"},
    {"component": "primary loss", "definition": "one-half mean squared error during optimization"},
    {"component": "reporting metrics", "definition": "RMSE, MAE, and R-squared on future periods"},
    {"component": "evaluation boundary", "definition": "chronological train, validation, and test split"},
])

display(decision_contract_df)

In [ ]:
# ============================================================
# 2.2 Generate synthetic campaign data
# ============================================================
def generate_campaign_data(n=N_CAMPAIGNS, seed=SEED):
    rng = np.random.default_rng(seed)
    day = np.arange(n)
    trend = day / max(n - 1, 1)
    date = pd.Timestamp("2024-01-01") + pd.to_timedelta(day, unit="D")

    channel = rng.choice(
        ["search", "social", "email", "display"],
        size=n,
        p=[0.32, 0.28, 0.24, 0.16],
    )
    channel_effect_map = {"search": 1.2, "social": 0.5, "email": 0.9, "display": -0.4}
    channel_effect = np.array([channel_effect_map[value] for value in channel])

    ad_spend = rng.gamma(shape=2.6, scale=1150, size=n) * (1 + 0.35 * trend)
    audience_size = rng.lognormal(mean=np.log(36000), sigma=0.42, size=n) * (1 + 0.55 * trend)
    discount_rate = np.clip(rng.beta(2.0, 7.5, size=n) * 0.48 + 0.025 * trend, 0, 0.45)
    season_index = np.sin(2 * np.pi * day / 30) + 0.35 * np.cos(2 * np.pi * day / 90)
    competitor_price_index = rng.normal(loc=1.0 + 0.035 * trend, scale=0.045, size=n)

    expected_revenue_k = (
        5.5
        + 0.00175 * ad_spend
        + 0.000085 * audience_size
        + 17.0 * discount_rate
        + 1.7 * season_index
        - 5.2 * (competitor_price_index - 1.0)
        + channel_effect
    )
    revenue_next_7d_k = expected_revenue_k + rng.normal(0, 1.45, size=n)

    # A few rare measurement or operational shocks create outliers.
    outlier_mask = rng.random(n) < 0.018
    revenue_next_7d_k[outlier_mask] += rng.normal(0, 10.0, size=outlier_mask.sum())
    revenue_next_7d_k = np.clip(revenue_next_7d_k, 0.2, None)

    conversion_logit = (
        -1.4
        + 0.00023 * ad_spend
        + 3.2 * discount_rate
        + 0.55 * season_index
        - 2.8 * (competitor_price_index - 1.0)
        + 0.30 * (channel == "search")
        + 0.20 * (channel == "email")
    )
    conversion_probability = sigmoid(conversion_logit)
    conversion_next_7d = rng.binomial(1, conversion_probability)

    return pd.DataFrame({
        "campaign_id": [f"CMP-{value:04d}" for value in range(n)],
        "launch_date": date,
        "channel": channel,
        "ad_spend": ad_spend,
        "audience_size": audience_size,
        "discount_rate": discount_rate,
        "season_index": season_index,
        "competitor_price_index": competitor_price_index,
        "revenue_next_7d_k": revenue_next_7d_k,
        "conversion_next_7d": conversion_next_7d,
        "is_outlier": outlier_mask.astype(int),
    })


campaign_df = generate_campaign_data()
print("Shape:", campaign_df.shape)
print("Date range:", campaign_df["launch_date"].min().date(), "to", campaign_df["launch_date"].max().date())
display(campaign_df.head())

In [ ]:
# ============================================================
# 2.3 Sanity checks before optimization
# ============================================================
summary_df = campaign_df[[
    "ad_spend",
    "audience_size",
    "discount_rate",
    "season_index",
    "competitor_price_index",
    "revenue_next_7d_k",
]].describe().T
summary_df["scale_ratio_to_smallest_std"] = summary_df["std"] / summary_df["std"].min()

display(summary_df)
print("Conversion rate:", round(campaign_df["conversion_next_7d"].mean(), 3))
print("Outlier share:", round(campaign_df["is_outlier"].mean(), 3))

rolling_revenue = campaign_df.set_index("launch_date")["revenue_next_7d_k"].rolling(30, min_periods=10).mean()
plt.figure(figsize=(9, 4))
plt.plot(campaign_df["launch_date"], campaign_df["revenue_next_7d_k"], alpha=0.25, label="campaign revenue")
plt.plot(rolling_revenue.index, rolling_revenue.values, linewidth=2, label="30-day rolling mean")
plt.xlabel("Launch date")
plt.ylabel("Revenue next 7 days ($000s)")
plt.title("Synthetic campaign outcome with time drift and occasional outliers")
plt.legend()
plt.show()

## 3. Regression and classification loss functions

A loss converts prediction error into a number that the optimizer can minimize. The choice of loss is part of the business specification because it determines which mistakes receive the largest penalties.

Squared loss strongly penalizes large residuals and has a smooth derivative. Absolute loss grows linearly and is more resistant to outliers, but it has a sharp corner at zero. Huber loss is quadratic near zero and linear beyond a chosen threshold. For classification, cross-entropy loss rewards accurate probabilities and sharply penalizes confident errors. Hinge loss instead focuses on whether a signed score satisfies a margin.

In [ ]:
# ============================================================
# 3.1 Regression loss curves
# ============================================================
residual_grid = np.linspace(-5, 5, 500)
loss_curve_df = pd.DataFrame({
    "residual": residual_grid,
    "squared": squared_loss(residual_grid),
    "absolute": absolute_loss(residual_grid),
    "huber_delta_1": huber_loss(residual_grid, delta=1.0),
})

plt.figure(figsize=(8, 5))
plt.plot(loss_curve_df["residual"], loss_curve_df["squared"], label="squared loss")
plt.plot(loss_curve_df["residual"], loss_curve_df["absolute"], label="absolute loss")
plt.plot(loss_curve_df["residual"], loss_curve_df["huber_delta_1"], label="Huber loss, delta=1")
plt.axvline(-1, linestyle="--", linewidth=1)
plt.axvline(1, linestyle="--", linewidth=1)
plt.xlabel("Residual: prediction minus actual")
plt.ylabel("Loss")
plt.title("The loss function determines how strongly large errors are penalized")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 3.2 One outlier can dominate squared loss
# ============================================================
ordinary_residuals = np.array([-0.8, 0.4, -0.3, 0.9, -0.5, 0.2, 0.6, -0.7])
residuals_with_outlier = np.append(ordinary_residuals, 8.0)


def summarize_residual_penalties(residuals, label):
    squared = squared_loss(residuals)
    absolute = absolute_loss(residuals)
    huber = huber_loss(residuals, delta=1.0)
    return {
        "case": label,
        "n": len(residuals),
        "mean_squared_loss": squared.mean(),
        "mean_absolute_loss": absolute.mean(),
        "mean_huber_loss": huber.mean(),
        "largest_squared_loss_share": squared.max() / squared.sum(),
        "largest_absolute_loss_share": absolute.max() / absolute.sum(),
        "largest_huber_loss_share": huber.max() / huber.sum(),
    }


outlier_loss_df = pd.DataFrame([
    summarize_residual_penalties(ordinary_residuals, "ordinary residuals"),
    summarize_residual_penalties(residuals_with_outlier, "same residuals plus one large error"),
])

display(outlier_loss_df)
print("Squared loss makes the largest error dominate the objective. Huber loss limits that influence while remaining smooth near zero.")

In [ ]:
# ============================================================
# 3.3 Cross-entropy and hinge loss
# ============================================================
probability_grid = np.linspace(0.001, 0.999, 500)
loss_if_positive = binary_log_loss(np.ones_like(probability_grid), probability_grid)
loss_if_negative = binary_log_loss(np.zeros_like(probability_grid), probability_grid)

plt.figure(figsize=(8, 5))
plt.plot(probability_grid, loss_if_positive, label="true class = 1")
plt.plot(probability_grid, loss_if_negative, label="true class = 0")
plt.xlabel("Predicted probability of class 1")
plt.ylabel("Cross-entropy loss")
plt.title("Cross-entropy heavily penalizes confident wrong probabilities")
plt.legend()
plt.show()

classification_examples_df = pd.DataFrame([
    {"true_label": 1, "predicted_probability": 0.90},
    {"true_label": 1, "predicted_probability": 0.60},
    {"true_label": 1, "predicted_probability": 0.10},
    {"true_label": 1, "predicted_probability": 0.01},
    {"true_label": 0, "predicted_probability": 0.10},
    {"true_label": 0, "predicted_probability": 0.99},
])
classification_examples_df["log_loss"] = binary_log_loss(
    classification_examples_df["true_label"],
    classification_examples_df["predicted_probability"],
)

display(classification_examples_df)

margin_grid = np.linspace(-3, 3, 400)
hinge_for_positive = np.maximum(0, 1 - margin_grid)
plt.figure(figsize=(8, 4))
plt.plot(margin_grid, hinge_for_positive)
plt.axvline(1, linestyle="--", linewidth=1, label="margin satisfied")
plt.xlabel("Signed model score for a positive case")
plt.ylabel("Hinge loss")
plt.title("Hinge loss becomes zero after the margin is satisfied")
plt.legend()
plt.show()

## 4. Convexity, local minima, and saddle points

A convex objective has one global minimum, so every well-behaved downhill path leads to the same solution. A nonconvex objective can have several valleys. Different starting points may lead to different local minima even when the same update rule is used.

A saddle point is different from a local minimum. Its gradient can be zero even though the surface goes upward in one direction and downward in another. This is one reason that a near-zero gradient should be interpreted together with the loss history and the local geometry.

In [ ]:
# ============================================================
# 4.1 Convex and nonconvex loss landscapes
# ============================================================
def convex_objective(x):
    return (x - 1.5) ** 2 + 0.5


def nonconvex_objective(x):
    return 0.05 * x ** 4 - 0.8 * x ** 2 + 0.15 * x + 4.0


def nonconvex_gradient(x):
    return 0.20 * x ** 3 - 1.60 * x + 0.15


x_grid = np.linspace(-4, 4, 600)

plt.figure(figsize=(8, 4))
plt.plot(x_grid, convex_objective(x_grid))
plt.xlabel("Parameter value")
plt.ylabel("Objective")
plt.title("Convex objective: one global minimum")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(x_grid, nonconvex_objective(x_grid))
plt.xlabel("Parameter value")
plt.ylabel("Objective")
plt.title("Nonconvex objective: multiple valleys")
plt.show()

In [ ]:
# ============================================================
# 4.2 Different starting points can reach different local minima
# ============================================================
def run_one_dimensional_gd(objective, gradient, start, learning_rate, steps):
    value = float(start)
    rows = []
    for step in range(steps + 1):
        rows.append({"step": step, "parameter": value, "loss": float(objective(value))})
        value = value - learning_rate * float(gradient(value))
    return pd.DataFrame(rows)


nonconvex_runs = {}
for start in [-3.5, -0.8, 0.8, 3.5]:
    nonconvex_runs[start] = run_one_dimensional_gd(
        nonconvex_objective,
        nonconvex_gradient,
        start=start,
        learning_rate=0.08,
        steps=80,
    )

plt.figure(figsize=(8, 4))
plt.plot(x_grid, nonconvex_objective(x_grid), label="objective")
for start, trace in nonconvex_runs.items():
    plt.plot(trace["parameter"], trace["loss"], marker="o", markersize=2, linewidth=1, label=f"start={start}")
plt.xlabel("Parameter value")
plt.ylabel("Objective")
plt.title("Starting position can determine the final valley")
plt.legend()
plt.show()

local_minimum_df = pd.DataFrame([
    {
        "start": start,
        "final_parameter": trace.iloc[-1]["parameter"],
        "final_loss": trace.iloc[-1]["loss"],
    }
    for start, trace in nonconvex_runs.items()
])
display(local_minimum_df)

In [ ]:
# ============================================================
# 4.3 A zero gradient can also be a saddle point
# ============================================================
def saddle_objective(point):
    x, y = point
    return x ** 2 - y ** 2


def saddle_gradient(point):
    x, y = point
    return np.array([2 * x, -2 * y])


saddle_check_df = pd.DataFrame([
    {"point": "(0, 0)", "objective": saddle_objective((0, 0)), "gradient_norm": np.linalg.norm(saddle_gradient((0, 0)))},
    {"point": "(0.5, 0)", "objective": saddle_objective((0.5, 0)), "gradient_norm": np.linalg.norm(saddle_gradient((0.5, 0)))},
    {"point": "(0, 0.5)", "objective": saddle_objective((0, 0.5)), "gradient_norm": np.linalg.norm(saddle_gradient((0, 0.5)))},
])
display(saddle_check_df)

axis = np.linspace(-1.5, 1.5, 200)
xx, yy = np.meshgrid(axis, axis)
zz = xx ** 2 - yy ** 2
plt.figure(figsize=(6, 5))
plt.contour(xx, yy, zz, levels=20)
plt.scatter([0], [0], s=60, label="zero-gradient saddle")
plt.xlabel("Parameter 1")
plt.ylabel("Parameter 2")
plt.title("Saddle point: upward in one direction, downward in another")
plt.legend()
plt.show()

## 5. Derivatives, partial derivatives, and gradients

For one parameter, the derivative is the local slope. For several parameters, the gradient is a vector of partial derivatives. Each element says how the objective changes when one parameter moves while the others are held fixed.

The gradient points toward the steepest local increase. To minimize the objective, gradient descent moves in the opposite direction. Numerical finite differences provide a useful implementation check: a correctly coded analytical gradient should closely match the slope estimated by small perturbations.

In [ ]:
# ============================================================
# 5.1 A derivative is the slope of the tangent line
# ============================================================
def scalar_objective(theta):
    return (theta - 3.0) ** 2 + 2.0


def scalar_derivative(theta):
    return 2.0 * (theta - 3.0)


theta_0 = 0.5
slope_0 = scalar_derivative(theta_0)
tangent = scalar_objective(theta_0) + slope_0 * (x_grid - theta_0)

plt.figure(figsize=(8, 4))
plt.plot(x_grid, scalar_objective(x_grid), label="objective")
plt.plot(x_grid, tangent, linestyle="--", label=f"tangent at theta={theta_0}")
plt.scatter([theta_0], [scalar_objective(theta_0)], s=60)
plt.ylim(0, 25)
plt.xlabel("theta")
plt.ylabel("Objective")
plt.title("The derivative gives the local slope")
plt.legend()
plt.show()

print("Derivative at theta=0.5:", slope_0)
print("The derivative is negative, so increasing theta locally decreases the objective.")

In [ ]:
# ============================================================
# 5.2 Numerical approximation of a scalar derivative
# ============================================================
epsilon_rows = []
for epsilon in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8]:
    numerical = (
        scalar_objective(theta_0 + epsilon) - scalar_objective(theta_0 - epsilon)
    ) / (2 * epsilon)
    epsilon_rows.append({
        "epsilon": epsilon,
        "analytical_derivative": scalar_derivative(theta_0),
        "numerical_derivative": numerical,
        "absolute_error": abs(numerical - scalar_derivative(theta_0)),
    })

finite_difference_df = pd.DataFrame(epsilon_rows)
display(finite_difference_df)

In [ ]:
# ============================================================
# 5.3 A two-parameter gradient
# ============================================================
def two_parameter_objective(parameters):
    w1, w2 = parameters
    return 0.5 * (4.0 * (w1 - 1.0) ** 2 + 0.6 * (w2 + 2.0) ** 2)


def two_parameter_gradient(parameters):
    w1, w2 = parameters
    return np.array([4.0 * (w1 - 1.0), 0.6 * (w2 + 2.0)])


check_point = np.array([-1.5, 1.0])
analytical_gradient = two_parameter_gradient(check_point)
numerical_gradient = finite_difference_gradient(two_parameter_objective, check_point)

gradient_check_df = pd.DataFrame({
    "parameter": ["w1", "w2"],
    "analytical_gradient": analytical_gradient,
    "numerical_gradient": numerical_gradient,
    "absolute_difference": np.abs(analytical_gradient - numerical_gradient),
})
display(gradient_check_df)

w1_axis = np.linspace(-2.5, 3.5, 180)
w2_axis = np.linspace(-5.0, 2.0, 180)
w1_mesh, w2_mesh = np.meshgrid(w1_axis, w2_axis)
objective_mesh = 0.5 * (4.0 * (w1_mesh - 1.0) ** 2 + 0.6 * (w2_mesh + 2.0) ** 2)

sample_points = np.array([[-1.5, 1.0], [0.0, -4.0], [2.5, 0.5]])
sample_gradients = np.array([two_parameter_gradient(point) for point in sample_points])

plt.figure(figsize=(7, 5))
plt.contour(w1_mesh, w2_mesh, objective_mesh, levels=18)
plt.quiver(
    sample_points[:, 0],
    sample_points[:, 1],
    sample_gradients[:, 0],
    sample_gradients[:, 1],
    angles="xy",
    scale_units="xy",
    scale=7,
    label="gradient direction",
)
plt.scatter([1.0], [-2.0], s=70, label="global minimum")
plt.xlabel("w1")
plt.ylabel("w2")
plt.title("A gradient contains one partial derivative per parameter")
plt.legend()
plt.show()

## 6. The gradient descent update rule and learning rate

The update rule is

\[
\theta_{new} = \theta_{old} - \alpha \nabla C(\theta_{old}),
\]

where \(\alpha\) is the learning rate. A very small learning rate produces slow progress. A useful learning rate decreases the loss quickly without instability. A learning rate that is too large overshoots the minimum and can oscillate or diverge.

In [ ]:
# ============================================================
# 6.1 Apply the update rule to a one-dimensional objective
# ============================================================
learning_rate_runs = {}
for learning_rate in [0.05, 0.45, 1.10]:
    learning_rate_runs[learning_rate] = run_one_dimensional_gd(
        scalar_objective,
        scalar_derivative,
        start=-4.0,
        learning_rate=learning_rate,
        steps=14,
    )

learning_rate_summary_df = pd.DataFrame([
    {
        "learning_rate": learning_rate,
        "initial_loss": trace.iloc[0]["loss"],
        "final_parameter": trace.iloc[-1]["parameter"],
        "final_loss": trace.iloc[-1]["loss"],
        "loss_decreased": trace.iloc[-1]["loss"] < trace.iloc[0]["loss"],
    }
    for learning_rate, trace in learning_rate_runs.items()
])
display(learning_rate_summary_df)

In [ ]:
# ============================================================
# 6.2 Visualize slow, effective, and unstable learning rates
# ============================================================
plt.figure(figsize=(8, 4))
for learning_rate, trace in learning_rate_runs.items():
    plt.plot(trace["step"], trace["loss"], marker="o", label=f"alpha={learning_rate}")
plt.yscale("log")
plt.xlabel("Update step")
plt.ylabel("Objective, log scale")
plt.title("Learning rate controls speed and stability")
plt.legend()
plt.show()

for learning_rate, trace in learning_rate_runs.items():
    plt.figure(figsize=(8, 4))
    plt.plot(x_grid, scalar_objective(x_grid), label="objective")
    plt.plot(trace["parameter"], trace["loss"], marker="o", label=f"gradient descent, alpha={learning_rate}")
    plt.xlabel("theta")
    plt.ylabel("Objective")
    plt.title(f"Gradient descent path with learning rate {learning_rate}")
    plt.ylim(0, min(120, max(25, trace["loss"].replace([np.inf, -np.inf], np.nan).dropna().max() * 1.1)))
    plt.legend()
    plt.show()

## 7. Linear regression from scratch

The campaign model uses five numeric predictors. The split follows time: the earliest 60 percent of campaigns are training data, the next 20 percent form the validation period, and the final 20 percent are held out for testing.

Standardization is fitted on the training period and then reused without recalculation. The design matrix adds a leading column of ones for the intercept. For one-half mean squared error, the gradient is

\[
\nabla C(w) = \frac{1}{n}X^T(Xw-y).
\]

In [ ]:
# ============================================================
# 7.1 Chronological split and training-only standardization
# ============================================================
FEATURES = [
    "ad_spend",
    "audience_size",
    "discount_rate",
    "season_index",
    "competitor_price_index",
]
TARGET = "revenue_next_7d_k"

n_total = len(campaign_df)
train_end = int(0.60 * n_total)
validation_end = int(0.80 * n_total)

train_df = campaign_df.iloc[:train_end].copy()
validation_df = campaign_df.iloc[train_end:validation_end].copy()
test_df = campaign_df.iloc[validation_end:].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[FEATURES])
X_validation_scaled = scaler.transform(validation_df[FEATURES])
X_test_scaled = scaler.transform(test_df[FEATURES])

X_train = add_intercept(X_train_scaled)
X_validation = add_intercept(X_validation_scaled)
X_test = add_intercept(X_test_scaled)

y_train = train_df[TARGET].to_numpy(dtype=float)
y_validation = validation_df[TARGET].to_numpy(dtype=float)
y_test = test_df[TARGET].to_numpy(dtype=float)

split_summary_df = pd.DataFrame([
    {"split": "train", "n": len(train_df), "start": train_df["launch_date"].min(), "end": train_df["launch_date"].max(), "mean_target": y_train.mean()},
    {"split": "validation", "n": len(validation_df), "start": validation_df["launch_date"].min(), "end": validation_df["launch_date"].max(), "mean_target": y_validation.mean()},
    {"split": "test", "n": len(test_df), "start": test_df["launch_date"].min(), "end": test_df["launch_date"].max(), "mean_target": y_test.mean()},
])

display(split_summary_df)
print("Design matrix shape:", X_train.shape)
print("Feature means after training transformation:", np.round(X_train_scaled.mean(axis=0), 6))
print("Feature standard deviations after training transformation:", np.round(X_train_scaled.std(axis=0), 6))

In [ ]:
# ============================================================
# 7.2 Inspect one full-batch gradient update by hand
# ============================================================
initial_weights = np.zeros(X_train.shape[1])
initial_prediction = X_train @ initial_weights
initial_loss = half_mse(y_train, initial_prediction)
initial_gradient = linear_gradient(X_train, y_train, initial_weights)
DEMO_LEARNING_RATE = 0.08
updated_weights = initial_weights - DEMO_LEARNING_RATE * initial_gradient
updated_loss = half_mse(y_train, X_train @ updated_weights)

one_step_df = pd.DataFrame({
    "term": ["intercept"] + FEATURES,
    "initial_weight": initial_weights,
    "gradient": initial_gradient,
    "updated_weight": updated_weights,
})

display(one_step_df)
print("Initial loss:", round(initial_loss, 4))
print("Loss after one update:", round(updated_loss, 4))

In [ ]:
# ============================================================
# 7.3 Reusable linear optimizer with batches and validation monitoring
# ============================================================
def fit_linear_optimizer(
    X_train,
    y_train,
    X_validation=None,
    y_validation=None,
    epochs=200,
    batch_size=None,
    learning_rate=0.05,
    optimizer="sgd",
    momentum=0.9,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8,
    learning_rate_decay=0.0,
    l2=0.0,
    patience=None,
    seed=SEED,
    initial_weights=None,
    record_updates=False,
    max_update_records=600,
):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    n, d = X_train.shape
    batch_size = n if batch_size is None else int(max(1, min(batch_size, n)))

    rng = np.random.default_rng(seed)
    weights = np.zeros(d) if initial_weights is None else np.asarray(initial_weights, dtype=float).copy()
    velocity = np.zeros(d)
    first_moment = np.zeros(d)
    second_moment = np.zeros(d)

    history = []
    update_records = []
    best_weights = weights.copy()
    best_validation_loss = np.inf
    best_epoch = -1
    wait = 0
    global_step = 0
    stop_reason = "maximum epochs reached"

    for epoch in range(int(epochs)):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            batch_indices = order[start:start + batch_size]
            X_batch = X_train[batch_indices]
            y_batch = y_train[batch_indices]
            gradient = linear_gradient(X_batch, y_batch, weights, l2=l2)
            current_learning_rate = learning_rate / (1.0 + learning_rate_decay * global_step)

            if optimizer == "sgd":
                weights = weights - current_learning_rate * gradient
            elif optimizer == "momentum":
                velocity = momentum * velocity + gradient
                weights = weights - current_learning_rate * velocity
            elif optimizer == "adam":
                first_moment = beta1 * first_moment + (1 - beta1) * gradient
                second_moment = beta2 * second_moment + (1 - beta2) * (gradient ** 2)
                corrected_first = first_moment / (1 - beta1 ** (global_step + 1))
                corrected_second = second_moment / (1 - beta2 ** (global_step + 1))
                weights = weights - current_learning_rate * corrected_first / (np.sqrt(corrected_second) + epsilon)
            else:
                raise ValueError("optimizer must be 'sgd', 'momentum', or 'adam'")

            if record_updates and len(update_records) < max_update_records:
                update_records.append({
                    "update": global_step,
                    "epoch": epoch,
                    "batch_loss": half_mse(y_batch, X_batch @ weights),
                    "gradient_norm": float(np.linalg.norm(gradient)),
                    "learning_rate": float(current_learning_rate),
                })
            global_step += 1

        train_data_loss = half_mse(y_train, X_train @ weights)
        penalty_weights = weights.copy()
        penalty_weights[0] = 0.0
        train_objective = train_data_loss + 0.5 * l2 * float(np.sum(penalty_weights ** 2))
        full_gradient_norm = float(np.linalg.norm(linear_gradient(X_train, y_train, weights, l2=l2)))

        validation_loss = np.nan
        if X_validation is not None and y_validation is not None:
            validation_loss = half_mse(y_validation, np.asarray(X_validation) @ weights)

        history.append({
            "epoch": epoch + 1,
            "train_data_loss": train_data_loss,
            "train_objective": train_objective,
            "validation_loss": validation_loss,
            "gradient_norm": full_gradient_norm,
            "weight_norm": float(np.linalg.norm(weights[1:])),
            "updates_completed": global_step,
        })

        if not np.isfinite(train_objective) or not np.all(np.isfinite(weights)):
            stop_reason = "non-finite loss or weights"
            break

        if X_validation is not None and y_validation is not None:
            if validation_loss < best_validation_loss - 1e-10:
                best_validation_loss = validation_loss
                best_weights = weights.copy()
                best_epoch = epoch + 1
                wait = 0
            else:
                wait += 1

            if patience is not None and wait >= patience:
                stop_reason = f"early stopping after {patience} epochs without improvement"
                weights = best_weights.copy()
                break

    history_df = pd.DataFrame(history)
    updates_df = pd.DataFrame(update_records)

    if best_epoch < 0 and len(history_df) > 0:
        best_epoch = int(history_df.loc[history_df["train_data_loss"].idxmin(), "epoch"])
        best_validation_loss = np.nan

    summary = {
        "optimizer": optimizer,
        "epochs_ran": int(len(history_df)),
        "updates_completed": int(global_step),
        "batch_size": int(batch_size),
        "learning_rate": float(learning_rate),
        "learning_rate_decay": float(learning_rate_decay),
        "l2": float(l2),
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_validation_loss) if np.isfinite(best_validation_loss) else np.nan,
        "stop_reason": stop_reason,
    }
    return weights, history_df, updates_df, summary

In [ ]:
# ============================================================
# 7.4 Full-batch gradient descent
# ============================================================
start_time = time.perf_counter()
full_batch_weights, full_batch_history_df, _, full_batch_summary = fit_linear_optimizer(
    X_train,
    y_train,
    X_validation,
    y_validation,
    epochs=MAX_EPOCHS,
    batch_size=len(X_train),
    learning_rate=0.08,
    optimizer="sgd",
    seed=SEED,
)
full_batch_seconds = time.perf_counter() - start_time

full_batch_validation_prediction = X_validation @ full_batch_weights
full_batch_test_prediction = X_test @ full_batch_weights
full_batch_metrics_df = pd.DataFrame([
    {"split": "validation", **regression_metrics(y_validation, full_batch_validation_prediction)},
    {"split": "test", **regression_metrics(y_test, full_batch_test_prediction)},
])

plt.figure(figsize=(8, 4))
plt.plot(full_batch_history_df["epoch"], full_batch_history_df["train_data_loss"], label="training loss")
plt.plot(full_batch_history_df["epoch"], full_batch_history_df["validation_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.yscale("log")
plt.ylabel("One-half mean squared error, log scale")
plt.title("Full-batch gradient descent converges smoothly")
plt.legend()
plt.show()

display(full_batch_metrics_df)
print({**full_batch_summary, "fit_seconds": round(full_batch_seconds, 4)})

In [ ]:
# ============================================================
# 7.5 Compare gradient descent with closed-form solutions
# ============================================================
closed_form_weights = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
closed_form_test_prediction = X_test @ closed_form_weights

sklearn_linear = LinearRegression()
sklearn_linear.fit(train_df[FEATURES], y_train)
sklearn_test_prediction = sklearn_linear.predict(test_df[FEATURES])

solution_comparison_df = pd.DataFrame([
    {"solution": "manual full-batch gradient descent", **regression_metrics(y_test, full_batch_test_prediction)},
    {"solution": "NumPy least squares", **regression_metrics(y_test, closed_form_test_prediction)},
    {"solution": "scikit-learn LinearRegression", **regression_metrics(y_test, sklearn_test_prediction)},
])

coefficient_comparison_df = standardized_to_raw_coefficients(full_batch_weights, scaler, FEATURES)
closed_form_raw_df = standardized_to_raw_coefficients(closed_form_weights, scaler, FEATURES).rename(
    columns={"coefficient": "closed_form_coefficient"}
)
coefficient_comparison_df = coefficient_comparison_df.rename(columns={"coefficient": "gradient_descent_coefficient"})
coefficient_comparison_df = coefficient_comparison_df.merge(closed_form_raw_df, on="term")
coefficient_comparison_df["absolute_difference"] = np.abs(
    coefficient_comparison_df["gradient_descent_coefficient"]
    - coefficient_comparison_df["closed_form_coefficient"]
)

display(solution_comparison_df)
display(coefficient_comparison_df)
print("Maximum absolute difference in test predictions:", round(np.max(np.abs(full_batch_test_prediction - closed_form_test_prediction)), 8))

In [ ]:
# ============================================================
# 7.6 Gradient check for the linear regression implementation
# ============================================================
check_weights = np.linspace(-0.25, 0.25, X_train.shape[1])


def training_objective_for_check(weights):
    return half_mse(y_train, X_train @ weights)


analytical_linear_gradient = linear_gradient(X_train, y_train, check_weights)
numerical_linear_gradient = finite_difference_gradient(
    training_objective_for_check,
    check_weights,
    epsilon=1e-6,
)

linear_gradient_check_df = pd.DataFrame({
    "term": ["intercept"] + FEATURES,
    "analytical_gradient": analytical_linear_gradient,
    "numerical_gradient": numerical_linear_gradient,
    "absolute_difference": np.abs(analytical_linear_gradient - numerical_linear_gradient),
})

display(linear_gradient_check_df)
print("Maximum absolute gradient difference:", linear_gradient_check_df["absolute_difference"].max())

## 8. Full-batch, stochastic, and mini-batch gradient descent

Full-batch gradient descent uses every training row for each update. Its gradient is stable, but each update becomes expensive as the dataset grows. Stochastic gradient descent uses one row at a time. Its updates are cheap but noisy. Mini-batch gradient descent averages a small group of rows and usually provides a practical compromise.

An epoch means that the optimizer has processed approximately one full pass through the training data. The number of updates per epoch therefore depends on batch size.

In [ ]:
# ============================================================
# 8.1 Quantify gradient noise at the initial weights
# ============================================================
rng = np.random.default_rng(SEED)
reference_gradient = linear_gradient(X_train, y_train, np.zeros(X_train.shape[1]))

gradient_noise_rows = []
for batch_size in [1, 8, 32, 128, len(X_train)]:
    errors = []
    gradient_norms = []
    repetitions = 1 if batch_size == len(X_train) else 250
    for _ in range(repetitions):
        if batch_size == len(X_train):
            batch_indices = np.arange(len(X_train))
        else:
            batch_indices = rng.choice(len(X_train), size=batch_size, replace=False)
        estimate = linear_gradient(
            X_train[batch_indices],
            y_train[batch_indices],
            np.zeros(X_train.shape[1]),
        )
        errors.append(np.linalg.norm(estimate - reference_gradient))
        gradient_norms.append(np.linalg.norm(estimate))
    gradient_noise_rows.append({
        "batch_size": batch_size,
        "updates_per_epoch": int(math.ceil(len(X_train) / batch_size)),
        "mean_gradient_error": np.mean(errors),
        "std_gradient_error": np.std(errors),
        "mean_gradient_norm": np.mean(gradient_norms),
    })

gradient_noise_df = pd.DataFrame(gradient_noise_rows)
display(gradient_noise_df)

plt.figure(figsize=(7, 4))
plt.plot(gradient_noise_df["batch_size"], gradient_noise_df["mean_gradient_error"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Batch size, log scale")
plt.ylabel("Mean distance from full gradient, log scale")
plt.title("Larger batches produce less noisy gradient estimates")
plt.show()

In [ ]:
# ============================================================
# 8.2 Train with three batch strategies
# ============================================================
batch_configs = [
    {"name": "full-batch", "batch_size": len(X_train), "learning_rate": 0.08},
    {"name": "stochastic", "batch_size": 1, "learning_rate": 0.004},
    {"name": "mini-batch 32", "batch_size": 32, "learning_rate": 0.025},
]

batch_histories = {}
batch_update_traces = {}
batch_weights = {}
batch_summary_rows = []

for config in batch_configs:
    start_time = time.perf_counter()
    weights, history_df, updates_df, summary = fit_linear_optimizer(
        X_train,
        y_train,
        X_validation,
        y_validation,
        epochs=45,
        batch_size=config["batch_size"],
        learning_rate=config["learning_rate"],
        optimizer="sgd",
        seed=SEED,
        record_updates=True,
    )
    elapsed = time.perf_counter() - start_time
    prediction = X_validation @ weights
    batch_histories[config["name"]] = history_df
    batch_update_traces[config["name"]] = updates_df
    batch_weights[config["name"]] = weights
    batch_summary_rows.append({
        "strategy": config["name"],
        "batch_size": config["batch_size"],
        "learning_rate": config["learning_rate"],
        "updates_per_epoch": int(math.ceil(len(X_train) / config["batch_size"])),
        "fit_seconds": elapsed,
        **regression_metrics(y_validation, prediction),
        "final_gradient_norm": history_df.iloc[-1]["gradient_norm"],
    })

batch_summary_df = pd.DataFrame(batch_summary_rows)
display(batch_summary_df)

In [ ]:
# ============================================================
# 8.3 Smoothness, update count, and noise
# ============================================================
plt.figure(figsize=(8, 4))
for name, history_df in batch_histories.items():
    plt.plot(history_df["epoch"], history_df["validation_loss"], label=name)
plt.xlabel("Epoch")
plt.yscale("log")
plt.ylabel("Validation one-half MSE, log scale")
plt.title("Batch strategy changes the optimization path")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for name, updates_df in batch_update_traces.items():
    if len(updates_df) == 0:
        continue
    plt.plot(updates_df["update"], updates_df["batch_loss"], alpha=0.75, label=name)
plt.yscale("log")
plt.xlabel("Update number")
plt.ylabel("Current batch loss, log scale")
plt.title("Small batches create noisier update-level losses")
plt.legend()
plt.show()

## 9. Feature scaling and leakage-safe preprocessing

Feature scaling changes the geometry of the objective. When one feature varies by tens of thousands and another varies between zero and one, the contours can become long, narrow ellipses. Gradient descent then makes rapid progress in one direction but extremely slow progress in another.

Min-max normalization maps a feature to a fixed range, commonly zero to one:

\[
x' = \frac{x - \min(X)}{\max(X) - \min(X)}.
\]

Standardization centers a feature at zero and scales it to unit standard deviation:

\[
z = \frac{x - \mu}{\sigma}.
\]

Min-max normalization is especially sensitive to extreme minimum and maximum values. Standardization is not immune to outliers, but it does not force every observation into a fixed interval. Both methods must be fitted on training data only.

The condition number is the ratio between the largest and smallest curvature directions. A large condition number indicates an elongated objective and slow first-order optimization. Standardization often reduces that ratio. Using validation or test statistics is leakage even when the resulting score does not improve.

In [ ]:
# ============================================================
# 9.1 Normalization and standardization react differently to an outlier
# ============================================================
scaling_demo_values = np.array([1000, 1500, 2000, 2500, 3000, 20000], dtype=float).reshape(-1, 1)
minmax_demo = MinMaxScaler().fit_transform(scaling_demo_values).ravel()
standard_demo = StandardScaler().fit_transform(scaling_demo_values).ravel()

outlier_scaling_df = pd.DataFrame({
    "original_value": scaling_demo_values.ravel(),
    "min_max_normalized": minmax_demo,
    "standardized": standard_demo,
    "ordinary_value": [True, True, True, True, True, False],
})

ordinary_rows = outlier_scaling_df[outlier_scaling_df["ordinary_value"]]
scaling_method_df = pd.DataFrame([
    {
        "method": "min-max normalization",
        "full_transformed_min": outlier_scaling_df["min_max_normalized"].min(),
        "full_transformed_max": outlier_scaling_df["min_max_normalized"].max(),
        "span_among_ordinary_values": ordinary_rows["min_max_normalized"].max() - ordinary_rows["min_max_normalized"].min(),
    },
    {
        "method": "standardization",
        "full_transformed_min": outlier_scaling_df["standardized"].min(),
        "full_transformed_max": outlier_scaling_df["standardized"].max(),
        "span_among_ordinary_values": ordinary_rows["standardized"].max() - ordinary_rows["standardized"].min(),
    },
])

display(outlier_scaling_df)
display(scaling_method_df)
print("The extreme value compresses the ordinary observations into a narrow portion of the min-max range.")

In [ ]:
# ============================================================
# 9.2 Curvature and condition number before and after scaling
# ============================================================
scaling_features = ["ad_spend", "discount_rate"]
X_two_raw = train_df[scaling_features].to_numpy(dtype=float)
X_two_raw_centered = X_two_raw - X_two_raw.mean(axis=0)
y_two_centered = y_train - y_train.mean()

X_two_standardized = StandardScaler().fit_transform(X_two_raw)


def curvature_summary(X, label):
    hessian = X.T @ X / len(X)
    eigenvalues = np.linalg.eigvalsh(hessian)
    return {
        "representation": label,
        "smallest_eigenvalue": eigenvalues.min(),
        "largest_eigenvalue": eigenvalues.max(),
        "condition_number": eigenvalues.max() / eigenvalues.min(),
        "safe_reference_learning_rate": 1.0 / eigenvalues.max(),
    }


curvature_df = pd.DataFrame([
    curvature_summary(X_two_raw_centered, "raw centered features"),
    curvature_summary(X_two_standardized, "standardized features"),
])
display(curvature_df)

In [ ]:
# ============================================================
# 9.3 Compare convergence on elongated and rounded objectives
# ============================================================
def run_quadratic_gradient_descent(X, y, iterations=300):
    hessian = X.T @ X / len(X)
    learning_rate = 1.0 / np.linalg.eigvalsh(hessian).max()
    optimum = np.linalg.lstsq(X, y, rcond=None)[0]
    optimum_loss = half_mse(y, X @ optimum)
    weights = np.zeros(X.shape[1])
    rows = []
    for iteration in range(iterations + 1):
        current_loss = half_mse(y, X @ weights)
        rows.append({
            "iteration": iteration,
            "loss": current_loss,
            "excess_loss": max(current_loss - optimum_loss, 1e-16),
            "distance_to_optimum": np.linalg.norm(weights - optimum),
            "w1": weights[0],
            "w2": weights[1],
        })
        gradient = X.T @ (X @ weights - y) / len(y)
        weights = weights - learning_rate * gradient
    return pd.DataFrame(rows), optimum, learning_rate


raw_scaling_trace_df, raw_scaling_optimum, raw_reference_lr = run_quadratic_gradient_descent(
    X_two_raw_centered,
    y_two_centered,
    iterations=350,
)
standardized_scaling_trace_df, standardized_scaling_optimum, standardized_reference_lr = run_quadratic_gradient_descent(
    X_two_standardized,
    y_two_centered,
    iterations=350,
)

plt.figure(figsize=(8, 4))
plt.plot(raw_scaling_trace_df["iteration"], raw_scaling_trace_df["excess_loss"], label="raw features")
plt.plot(standardized_scaling_trace_df["iteration"], standardized_scaling_trace_df["excess_loss"], label="standardized features")
plt.yscale("log")
plt.xlabel("Gradient descent iteration")
plt.ylabel("Loss above optimum, log scale")
plt.title("Standardization can dramatically accelerate convergence")
plt.legend()
plt.show()

scaling_convergence_df = pd.DataFrame([
    {
        "representation": "raw centered features",
        "learning_rate": raw_reference_lr,
        "initial_excess_loss": raw_scaling_trace_df.iloc[0]["excess_loss"],
        "final_excess_loss": raw_scaling_trace_df.iloc[-1]["excess_loss"],
        "final_distance_to_optimum": raw_scaling_trace_df.iloc[-1]["distance_to_optimum"],
    },
    {
        "representation": "standardized features",
        "learning_rate": standardized_reference_lr,
        "initial_excess_loss": standardized_scaling_trace_df.iloc[0]["excess_loss"],
        "final_excess_loss": standardized_scaling_trace_df.iloc[-1]["excess_loss"],
        "final_distance_to_optimum": standardized_scaling_trace_df.iloc[-1]["distance_to_optimum"],
    },
])
display(scaling_convergence_df)

In [ ]:
# ============================================================
# 9.4 Reproduce the zigzag-versus-direct-path intuition
# ============================================================
def contour_with_path(X, y, trace_df, optimum, title):
    path = trace_df.iloc[:80]
    w1_values = np.concatenate([path["w1"].to_numpy(), [optimum[0]]])
    w2_values = np.concatenate([path["w2"].to_numpy(), [optimum[1]]])
    w1_margin = max(np.ptp(w1_values), 1e-6) * 0.35 + 1e-6
    w2_margin = max(np.ptp(w2_values), 1e-6) * 0.35 + 1e-6
    w1_axis = np.linspace(w1_values.min() - w1_margin, w1_values.max() + w1_margin, 160)
    w2_axis = np.linspace(w2_values.min() - w2_margin, w2_values.max() + w2_margin, 160)
    w1_mesh, w2_mesh = np.meshgrid(w1_axis, w2_axis)
    grid_weights = np.column_stack([w1_mesh.ravel(), w2_mesh.ravel()])
    predictions = X @ grid_weights.T
    losses = 0.5 * np.mean((predictions - y[:, None]) ** 2, axis=0).reshape(w1_mesh.shape)

    plt.figure(figsize=(7, 5))
    plt.contour(w1_mesh, w2_mesh, losses, levels=18)
    plt.plot(path["w1"], path["w2"], marker="o", markersize=2, linewidth=1, label="gradient path")
    plt.scatter([optimum[0]], [optimum[1]], s=70, label="optimum")
    plt.xlabel("Parameter 1")
    plt.ylabel("Parameter 2")
    plt.title(title)
    plt.legend()
    plt.show()


contour_with_path(
    X_two_raw_centered,
    y_two_centered,
    raw_scaling_trace_df,
    raw_scaling_optimum,
    "Raw features: elongated contours and slow progress",
)
contour_with_path(
    X_two_standardized,
    y_two_centered,
    standardized_scaling_trace_df,
    standardized_scaling_optimum,
    "Standardized features: rounder contours and direct progress",
)

In [ ]:
# ============================================================
# 9.5 Leakage audit: training-only versus full-data scaling
# ============================================================
training_only_scaler = StandardScaler().fit(train_df[FEATURES])
invalid_full_data_scaler = StandardScaler().fit(campaign_df[FEATURES])

scaling_parameter_rows = []
for j, feature in enumerate(FEATURES):
    scaling_parameter_rows.append({
        "feature": feature,
        "training_mean": training_only_scaler.mean_[j],
        "full_data_mean_including_future": invalid_full_data_scaler.mean_[j],
        "training_std": training_only_scaler.scale_[j],
        "full_data_std_including_future": invalid_full_data_scaler.scale_[j],
    })
scaling_parameter_audit_df = pd.DataFrame(scaling_parameter_rows)
display(scaling_parameter_audit_df)

scaling_workflow_rows = []
for workflow_name, fitted_scaler in [
    ("valid: scaler fit on training period", training_only_scaler),
    ("invalid: scaler fit on all periods", invalid_full_data_scaler),
]:
    X_train_variant = add_intercept(fitted_scaler.transform(train_df[FEATURES]))
    X_validation_variant = add_intercept(fitted_scaler.transform(validation_df[FEATURES]))
    X_test_variant = add_intercept(fitted_scaler.transform(test_df[FEATURES]))
    weights, history_df, _, _ = fit_linear_optimizer(
        X_train_variant,
        y_train,
        X_validation_variant,
        y_validation,
        epochs=120,
        batch_size=len(X_train_variant),
        learning_rate=0.08,
        optimizer="sgd",
        seed=SEED,
    )
    scaling_workflow_rows.append({
        "workflow": workflow_name,
        "validation_rmse": regression_metrics(y_validation, X_validation_variant @ weights)["rmse"],
        "test_rmse": regression_metrics(y_test, X_test_variant @ weights)["rmse"],
        "final_training_loss": history_df.iloc[-1]["train_data_loss"],
    })

scaling_workflow_df = pd.DataFrame(scaling_workflow_rows)
display(scaling_workflow_df)
print("The invalid workflow must be rejected regardless of whether its score is higher or lower. It uses information from future periods.")

## 10. Learning-rate schedules, momentum, and Adam

Plain stochastic gradient descent uses the current gradient directly. A learning-rate schedule makes updates smaller over time. Momentum accumulates a moving direction so that updates can continue through shallow directions and damp some zigzag behavior. Adam rescales each parameter using running estimates of the first and second moments of the gradient.

These methods do not remove the need for validation. Their settings are hyperparameters, and a faster drop in training loss does not automatically mean better future performance.

In [ ]:
# ============================================================
# 10.1 Compare basic optimizer variants on the same training data
# ============================================================
optimizer_configs = [
    {"name": "SGD", "optimizer": "sgd", "learning_rate": 0.025, "decay": 0.0},
    {"name": "SGD with decay", "optimizer": "sgd", "learning_rate": 0.035, "decay": 0.0015},
    {"name": "momentum", "optimizer": "momentum", "learning_rate": 0.005, "decay": 0.0},
    {"name": "Adam", "optimizer": "adam", "learning_rate": 0.080, "decay": 0.0},
]

optimizer_histories = {}
optimizer_weights = {}
optimizer_summary_rows = []

for config in optimizer_configs:
    start_time = time.perf_counter()
    weights, history_df, _, summary = fit_linear_optimizer(
        X_train,
        y_train,
        X_validation,
        y_validation,
        epochs=70,
        batch_size=32,
        learning_rate=config["learning_rate"],
        optimizer=config["optimizer"],
        learning_rate_decay=config["decay"],
        seed=SEED,
    )
    elapsed = time.perf_counter() - start_time
    optimizer_histories[config["name"]] = history_df
    optimizer_weights[config["name"]] = weights
    optimizer_summary_rows.append({
        "optimizer": config["name"],
        "initial_learning_rate": config["learning_rate"],
        "decay": config["decay"],
        "fit_seconds": elapsed,
        "final_train_loss": history_df.iloc[-1]["train_data_loss"],
        "final_validation_loss": history_df.iloc[-1]["validation_loss"],
        "final_gradient_norm": history_df.iloc[-1]["gradient_norm"],
        **regression_metrics(y_validation, X_validation @ weights),
    })

optimizer_summary_df = pd.DataFrame(optimizer_summary_rows).sort_values("rmse")
display(optimizer_summary_df)

In [ ]:
# ============================================================
# 10.2 Optimizer convergence curves
# ============================================================
plt.figure(figsize=(8, 4))
for name, history_df in optimizer_histories.items():
    plt.plot(history_df["epoch"], history_df["validation_loss"], label=name)
plt.xlabel("Epoch")
plt.yscale("log")
plt.ylabel("Validation one-half MSE, log scale")
plt.title("Optimizer choice changes the route, not the evaluation standard")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for name, history_df in optimizer_histories.items():
    plt.plot(history_df["epoch"], history_df["gradient_norm"], label=name)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Full training gradient norm, log scale")
plt.title("Gradient norm is one convergence diagnostic")
plt.legend()
plt.show()

## 11. Regularization and early stopping

Regularization changes the objective by penalizing large coefficients. Early stopping does not change the formula for the loss, but it stops optimization when validation performance has not improved for a specified number of epochs. Both methods can limit overfitting.

The controlled stress test below creates a small, high-dimensional training problem with noisy labels. The purpose is not to recommend polynomial regression for campaign forecasting. It is to make the difference between training loss and validation loss visible.

In [ ]:
# ============================================================
# 11.1 Build an intentionally over-parameterized stress test
# ============================================================
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_validation_poly = poly.transform(X_validation_scaled)
X_test_poly = poly.transform(X_test_scaled)

rng = np.random.default_rng(SEED + 17)
N_NOISE_FEATURES = 70 if FAST_MODE else 120
noise_train = rng.normal(size=(len(X_train_poly), N_NOISE_FEATURES))
noise_validation = rng.normal(size=(len(X_validation_poly), N_NOISE_FEATURES))
noise_test = rng.normal(size=(len(X_test_poly), N_NOISE_FEATURES))

X_train_complex_raw = np.column_stack([X_train_poly, noise_train])
X_validation_complex_raw = np.column_stack([X_validation_poly, noise_validation])
X_test_complex_raw = np.column_stack([X_test_poly, noise_test])

small_n = 110 if FAST_MODE else 180
small_indices = np.arange(small_n)
complex_scaler = StandardScaler().fit(X_train_complex_raw[small_indices])
X_small_complex = add_intercept(complex_scaler.transform(X_train_complex_raw[small_indices]))
X_validation_complex = add_intercept(complex_scaler.transform(X_validation_complex_raw))
X_test_complex = add_intercept(complex_scaler.transform(X_test_complex_raw))

y_small_noisy = y_train[small_indices].copy()
noisy_label_indices = rng.choice(small_n, size=max(10, small_n // 7), replace=False)
y_small_noisy[noisy_label_indices] += rng.normal(0, 7.0, size=len(noisy_label_indices))

stress_test_df = pd.DataFrame([
    {"quantity": "training rows", "value": len(X_small_complex)},
    {"quantity": "parameters including intercept", "value": X_small_complex.shape[1]},
    {"quantity": "corrupted training labels", "value": len(noisy_label_indices)},
    {"quantity": "validation rows", "value": len(X_validation_complex)},
])
display(stress_test_df)

In [ ]:
# ============================================================
# 11.2 Compare unrestricted training, early stopping, and L2
# ============================================================
regularization_configs = [
    {"name": "unregularized final epoch", "l2": 0.0, "patience": None},
    {"name": "early stopping", "l2": 0.0, "patience": 45},
    {"name": "L2 regularization", "l2": 1.00, "patience": None},
]

regularization_histories = {}
regularization_weights = {}
regularization_summary_rows = []

for config in regularization_configs:
    weights, history_df, _, summary = fit_linear_optimizer(
        X_small_complex,
        y_small_noisy,
        X_validation_complex,
        y_validation,
        epochs=850 if FAST_MODE else 1400,
        batch_size=len(X_small_complex),
        learning_rate=0.018,
        optimizer="sgd",
        l2=config["l2"],
        patience=config["patience"],
        seed=SEED,
    )
    regularization_histories[config["name"]] = history_df
    regularization_weights[config["name"]] = weights
    regularization_summary_rows.append({
        "strategy": config["name"],
        "epochs_ran": summary["epochs_ran"],
        "best_epoch": summary["best_epoch"],
        "l2": config["l2"],
        "coefficient_norm": np.linalg.norm(weights[1:]),
        "validation_rmse": regression_metrics(y_validation, X_validation_complex @ weights)["rmse"],
        "test_rmse": regression_metrics(y_test, X_test_complex @ weights)["rmse"],
        "stop_reason": summary["stop_reason"],
    })

regularization_summary_df = pd.DataFrame(regularization_summary_rows).sort_values("validation_rmse")
display(regularization_summary_df)

In [ ]:
# ============================================================
# 11.3 Training loss and validation loss answer different questions
# ============================================================
unregularized_history_df = regularization_histories["unregularized final epoch"]

plt.figure(figsize=(8, 4))
plt.plot(unregularized_history_df["epoch"], unregularized_history_df["train_data_loss"], label="training loss")
plt.plot(unregularized_history_df["epoch"], unregularized_history_df["validation_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("One-half MSE")
plt.title("More optimization can improve training fit while harming validation fit")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for name, history_df in regularization_histories.items():
    plt.plot(history_df["epoch"], history_df["validation_loss"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation one-half MSE")
plt.title("Early stopping and L2 limit overfitting in different ways")
plt.legend()
plt.show()

## 12. Troubleshooting and gradient checks

A training curve is a diagnostic signal, not a decorative figure. Divergence often indicates a learning rate that is too large, numerical overflow, or extreme feature scales. Very slow progress can indicate a learning rate that is too small or an ill-conditioned objective. Strong oscillation can arise from an excessive learning rate or very noisy batches. A falling training loss with a rising validation loss indicates overfitting rather than optimization failure.

In [ ]:
# ============================================================
# 12.1 A compact troubleshooting guide
# ============================================================
troubleshooting_df = pd.DataFrame([
    {
        "observed pattern": "loss becomes NaN, infinite, or rapidly larger",
        "likely causes": "learning rate too large; overflow; badly scaled features",
        "first actions": "reduce learning rate; standardize inputs; inspect extreme values",
    },
    {
        "observed pattern": "loss decreases very slowly",
        "likely causes": "learning rate too small; elongated objective; weak gradients",
        "first actions": "scale features; inspect condition number; cautiously increase learning rate",
    },
    {
        "observed pattern": "loss repeatedly jumps up and down",
        "likely causes": "learning rate too large; very small or unrepresentative batches",
        "first actions": "reduce learning rate; increase batch size; use a schedule or momentum",
    },
    {
        "observed pattern": "training loss falls while validation loss rises",
        "likely causes": "overfitting; noisy labels; excessive capacity",
        "first actions": "early stop; regularize; simplify features; investigate label quality",
    },
    {
        "observed pattern": "gradient is near zero but the objective is not satisfactory",
        "likely causes": "local minimum; saddle or plateau; saturated model component",
        "first actions": "try another initialization; inspect local geometry; reconsider the model",
    },
    {
        "observed pattern": "validation score looks unusually strong",
        "likely causes": "target leakage; future-aware preprocessing; test-driven tuning",
        "first actions": "audit feature timestamps, split logic, and preprocessing fit boundaries",
    },
])
display(troubleshooting_df)

In [ ]:
# ============================================================
# 12.2 Automatically summarize several observed traces
# ============================================================
def diagnose_loss_values(loss_values):
    values = np.asarray(loss_values, dtype=float)
    if len(values) < 3:
        return "too few observations"
    if not np.all(np.isfinite(values)):
        return "divergent or numerically unstable"
    if values[-1] > 1.5 * values[0]:
        return "loss increasing or diverging"
    minimum_index = int(np.argmin(values))
    minimum_value = float(values[minimum_index])
    late_minimum_cutoff = len(values) - max(5, int(0.15 * len(values)))
    if minimum_index < late_minimum_cutoff and values[-1] > 1.25 * max(minimum_value, 1e-12):
        return "validation deterioration after a minimum, possible overfitting"
    relative_improvement = (values[0] - values[-1]) / max(abs(values[0]), 1e-12)
    differences = np.diff(values)
    upward_fraction = np.mean(differences > 0)
    if upward_fraction > 0.35 and np.std(differences) > abs(np.mean(differences)):
        return "oscillating or noisy"
    if relative_improvement < 0.05:
        return "slow progress or plateau"
    return "generally healthy decline"


diagnostic_rows = []
for learning_rate, trace in learning_rate_runs.items():
    diagnostic_rows.append({
        "trace": f"scalar GD alpha={learning_rate}",
        "diagnosis": diagnose_loss_values(trace["loss"]),
    })
for name, history_df in batch_histories.items():
    diagnostic_rows.append({
        "trace": f"{name} validation loss",
        "diagnosis": diagnose_loss_values(history_df["validation_loss"]),
    })
diagnostic_rows.append({
    "trace": "over-parameterized validation loss",
    "diagnosis": diagnose_loss_values(unregularized_history_df["validation_loss"]),
})

trace_diagnostics_df = pd.DataFrame(diagnostic_rows)
display(trace_diagnostics_df)

In [ ]:
# ============================================================
# 12.3 Optimization audit checklist
# ============================================================
optimization_audit_df = pd.DataFrame([
    {"audit question": "Is the loss aligned with the target and business cost?", "current notebook": "documented: squared, absolute, Huber, and log loss are compared"},
    {"audit question": "Was the analytical gradient checked?", "current notebook": "pass: finite-difference check completed"},
    {"audit question": "Are training, validation, and test periods separated?", "current notebook": "pass: chronological 60/20/20 split"},
    {"audit question": "Was preprocessing fit on training data only?", "current notebook": "pass in main workflow; full-data scaling shown only as an invalid demonstration"},
    {"audit question": "Are learning rate, batch size, optimizer, and stopping rule recorded?", "current notebook": "pass: experiment summaries retain all four"},
    {"audit question": "Was validation used for tuning without touching the test set?", "current notebook": "pass: test results are reported after the workflow is defined"},
    {"audit question": "Were non-finite losses and unstable traces checked?", "current notebook": "pass: optimizer stops on non-finite values and traces are diagnosed"},
])
display(optimization_audit_df)

## 13. Governance and saved artifacts

An optimization result is reproducible only when the data boundary, objective, gradient, initialization, scaling policy, learning rate, batch size, optimizer, random seed, stopping rule, and evaluation period are recorded together. Saving only the final coefficients hides the path that produced them.

In [ ]:
# ============================================================
# 13.1 Optimization experiment card
# ============================================================
optimization_card = {
    "chapter": 4,
    "system_name": "synthetic_campaign_revenue_gradient_descent",
    "business_decision": "forecast next-seven-day campaign revenue for planning and budget review",
    "unit_of_analysis": "one campaign at its launch date",
    "target": TARGET,
    "features": FEATURES,
    "intended_use": "predictive planning, not causal attribution",
    "split_strategy": {
        "train": "earliest 60 percent of campaigns",
        "validation": "next 20 percent",
        "test": "final 20 percent",
    },
    "objective": "one-half mean squared error",
    "gradient": "X.T @ (X @ weights - y) / n",
    "initialization": "all zeros",
    "main_optimizer": "full-batch gradient descent",
    "main_learning_rate": 0.08,
    "main_epochs": MAX_EPOCHS,
    "scaling_methods_demonstrated": "min-max normalization and standardization",
    "scaling_policy": "StandardScaler fit on training data only and reused for validation and test",
    "closed_form_check": "NumPy least squares and scikit-learn LinearRegression",
    "gradient_check_max_absolute_difference": float(linear_gradient_check_df["absolute_difference"].max()),
    "main_test_metrics": regression_metrics(y_test, full_batch_test_prediction),
    "batch_strategies_compared": batch_summary_df.to_dict(orient="records"),
    "optimizer_variants_compared": optimizer_summary_df.to_dict(orient="records"),
    "leakage_control": "future-aware full-data scaling is displayed only as an invalid workflow",
    "known_limitations": [
        "the data are synthetic and simpler than a production marketing system",
        "the model is predictive and does not identify causal advertising effects",
        "rare outcome shocks are simulated rather than taken from operational records",
        "optimizer comparisons use compact classroom settings rather than exhaustive tuning",
    ],
}

print(json.dumps(optimization_card, indent=2))

In [ ]:
# ============================================================
# 13.2 Save shareable artifacts
# ============================================================
campaign_df.to_csv(OUT_DIR / "ch04_synthetic_campaign_data.csv", index=False)
full_batch_history_df.to_csv(OUT_DIR / "ch04_full_batch_history.csv", index=False)
full_batch_metrics_df.to_csv(OUT_DIR / "ch04_full_batch_metrics.csv", index=False)
coefficient_comparison_df.to_csv(OUT_DIR / "ch04_coefficient_comparison.csv", index=False)
gradient_noise_df.to_csv(OUT_DIR / "ch04_gradient_noise_by_batch_size.csv", index=False)
batch_summary_df.to_csv(OUT_DIR / "ch04_batch_strategy_summary.csv", index=False)
outlier_scaling_df.to_csv(OUT_DIR / "ch04_normalization_standardization_demo.csv", index=False)
curvature_df.to_csv(OUT_DIR / "ch04_scaling_curvature_summary.csv", index=False)
scaling_convergence_df.to_csv(OUT_DIR / "ch04_scaling_convergence_summary.csv", index=False)
scaling_parameter_audit_df.to_csv(OUT_DIR / "ch04_scaling_parameter_audit.csv", index=False)
scaling_workflow_df.to_csv(OUT_DIR / "ch04_scaling_workflow_audit.csv", index=False)
optimizer_summary_df.to_csv(OUT_DIR / "ch04_optimizer_summary.csv", index=False)
regularization_summary_df.to_csv(OUT_DIR / "ch04_regularization_early_stopping_summary.csv", index=False)
troubleshooting_df.to_csv(OUT_DIR / "ch04_troubleshooting_guide.csv", index=False)
optimization_audit_df.to_csv(OUT_DIR / "ch04_optimization_audit.csv", index=False)

np.save(OUT_DIR / "ch04_full_batch_weights_standardized.npy", full_batch_weights)
joblib.dump(scaler, OUT_DIR / "ch04_training_only_standard_scaler.joblib")

with open(OUT_DIR / "ch04_optimization_card.json", "w", encoding="utf-8") as file:
    json.dump(optimization_card, file, indent=2)

print("Saved artifacts:")
for path in sorted(OUT_DIR.iterdir()):
    print(" -", path)

## Decision guide

Begin with the target and the business meaning of error. Use squared loss when large errors deserve strong penalties and smooth optimization is useful. Consider absolute or Huber loss when rare extreme errors would otherwise dominate the objective. Use cross-entropy when a classification system must produce probabilities, and interpret hinge loss as a margin-based alternative rather than a probability score.

Standardize features when their scales differ substantially, and fit every transformation on the training data only. Start with a conservative learning rate and inspect both loss and gradient norms. Use full-batch updates for small, stable problems, mini-batches for scalable training, and stochastic updates when frequent noisy updates are acceptable. Momentum, schedules, and Adam can improve the path, but validation determines whether the result generalizes. Stop or regularize when validation performance deteriorates even though training loss continues to fall.

A loss that decreases is necessary but not sufficient. The optimizer must also respect the data timeline, reproduce under a fixed seed, and deliver useful performance on untouched future data.

## Exercises

1. Change the Huber threshold from `1.0` to `0.5` and `2.0`. Explain how the curve and the outlier penalty change.

2. For the scalar quadratic objective, test learning rates between `0.01` and `1.20`. Identify the fastest stable value and the first clearly divergent value.

3. Change the starting points in the nonconvex experiment. Map which starting values reach the left and right valleys.

4. Replace the chronological split with a random row split. Compare the test result and explain why the chronological split better matches future campaign deployment.

5. Train the manual linear regression without feature scaling. Search for a stable learning rate and compare the number of iterations required to reach the scaled model's loss.

6. Add `channel` by one-hot encoding it before gradient descent. Update the coefficient table and compare future-period RMSE.

7. Repeat the batch-size experiment with batch sizes `4`, `16`, `64`, and `256`. Plot gradient noise and validation loss.

8. Hold the mini-batch learning rate constant across all batch sizes. Explain why the same value may not be equally effective.

9. In the optimizer comparison, change momentum from `0.9` to `0.5` and `0.99`. Compare the convergence curves.

10. Increase the number of irrelevant noise features in the overfitting stress test. Record the best validation epoch and coefficient norm.

11. Replace the squared objective in the manual regression with Huber loss. Derive or implement its piecewise gradient and compare robustness to the simulated outcome shocks.

12. Add a stopping rule based on gradient norm. Compare it with validation-based early stopping and explain why the two rules answer different questions.

13. Deliberately fit the scaler on all periods, then write a one-paragraph audit finding that explains why the workflow is invalid even if its score is not better.

14. Add one optimization setting, one data-quality field, and one operational monitoring field to the optimization card.

## Wrap-up

This notebook converted gradient descent from a mountain analogy into a complete, inspectable optimization workflow. It compared loss functions, visualized convex and nonconvex landscapes, verified derivatives and gradients, demonstrated the update rule, trained linear regression from scratch, compared batch strategies and optimizers, quantified the effect of scaling, exposed preprocessing leakage, and used validation loss to control overfitting. The central lesson is that optimization is not a hidden mechanical step. It is a sequence of choices that should be measured, justified, and documented.